# Hands-on 1: from a saved model to a safe prediction
AICTE QIP · Artificial Intelligence and Engineering · IIIT Kottayam

You will do, step by step, what the live app does behind the screen.

**How to work:** run each cell with **Shift + Enter**. Change only the lines marked `# FILL` (replace `___`). Work with your partner.

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression

data = pd.read_csv("https://raw.githubusercontent.com/ansiths-iiitk/ews-demo/main/training_data.csv")   # made-up teaching data, not real students
data.head()

## Step 1 · Training (already done for you)
The model learns from 600 past students: attendance (%), internal marks (out of 100), and whether the student later needed review.

In [ ]:
model = LogisticRegression()
model.fit(data[["attendance", "marks"]], data["needs_review"])
print("Model trained on", len(data), "students")

## Step 2 · Save the trained model to a file
Training happens once, on the developer's machine. What comes out is just a file.

In [ ]:
joblib.dump(model, ___)          # FILL: a file name, for example "model.joblib"
print("File size:", os.path.getsize("model.joblib"), "bytes")

## Step 3 · Load it in a "different program"
Pretend this cell is the server program. It has never seen the training data; it only has the file.

In [ ]:
del model                                # the server starts with no model in memory
model = joblib.___("model.joblib")       # FILL: the opposite of dump
print(model)

## Step 4 · Inference: one prediction for one new student

In [ ]:
def predict(attendance, marks):
    row = pd.DataFrame([[attendance, marks]], columns=["attendance", "marks"])
    p = model.predict_proba(row)[0, 1]
    label = "Needs review" if p >= 0.5 else "On track"
    return f"{label}  (chance of needing review: {p:.0%})"

print(predict(85, 70))
print(predict(___, ___))                 # FILL: a student you expect to need review

## Step 5 · The inputs the room used to break the app
Run them on your own copy of the model.

In [ ]:
for a, m in [(150, 40), (-20, 90), (85, 1000)]:
    print(f"attendance {a:>4}, marks {m:>5}  ->  {predict(a, m)}")

**Discuss with your partner:** why does the model answer these with full confidence instead of complaining?

*Your answer:*

## Step 6 · Add input checking (validation)
Write the rules the room discovered. Then every request passes through `validate()` before it reaches the model.

In [ ]:
def validate(attendance, marks):
    """Return None if the inputs are acceptable, otherwise the reason for rejecting."""
    if attendance is None or marks is None:
        return "Both values are required."
    if not 0 <= attendance <= ___:        # FILL
        return "Attendance must be between 0 and 100."
    if not ___ <= marks <= 100:           # FILL
        return "Marks must be between 0 and 100."
    return None

def safe_predict(attendance, marks):
    problem = validate(attendance, marks)
    if problem:
        return "REJECTED: " + problem
    return predict(attendance, marks)

In [ ]:
for a, m in [(85, 70), (150, 40), (-20, 90), (85, 1000), (None, 50)]:
    print(f"attendance {str(a):>4}, marks {str(m):>5}  ->  {safe_predict(a, m)}")

## Step 7 · One more case (keep this in mind for the next part)
The college now gives internal marks **out of 20**. An excellent student scores 18 out of 20, and the form sends `18`.

In [ ]:
print(safe_predict(85, 18))

**Discuss:** did `validate()` stop this request? Is the answer right? What would have to watch the system to notice?

*Your answer:*